In [1]:
import pynamod
import nglview as nv
from pynamod.geometry.trajectories import open_h5

### Full Atomic Structure Analysis


In [2]:
nucl = pynamod.CG_Structure(pdb_id='3lz0')
# Also can be initialized with pdb filename or mda Universe
nucl.analyze_dna(leading_strands=['I'])
nucl.analyze_protein(n_cg_beads=80)

Sending GET request to https://files.rcsb.org/download/3lz0.pdb to fetch 3lz0's pdb file as a string.


In [3]:
nucl.view_structure()

NGLWidget()

In [4]:
# Or load existing file
nucl = pynamod.CG_Structure()
file = open_h5('cg_3lz0.h5', 'r')
nucl.load_from_h5(file)

<CG_Structure DNA_pairs=145 proteins=146>

### Linear DNA generation


In [5]:
seq = 'atcg'*4
dna_gen = pynamod.CG_Structure()
dna_gen.build_dna(seq)

In [6]:
dna_gen.view_structure()

NGLWidget()

### Structures merge


In [7]:
nucl.append_structures([dna_gen, nucl]*3)

<CG_Structure DNA_pairs=628 proteins=632>

In [8]:
nucl.view_structure()

NGLWidget()

In [9]:
nucl.dna.step_params

mod_Tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
            [-0.1033, -1.0316,  4.0098,  5.4315, -2.1597, 37.1837],
            [ 1.0071, -0.1374,  3.6875, -0.2604, -0.0614, 41.3648],
            ...,
            [ 0.1088,  0.5360,  2.7992, -5.8632,  9.5255, 27.9278],
            [-0.6006, -0.4360,  3.6323,  4.4464,  5.6871, 41.1649],
            [ 0.3301, -0.2588,  3.0272, -4.1746, -7.4785, 42.0669]],
           dtype=torch.float64)

In [10]:
nucl.dna.origins

mod_Tensor([[[  0.0000,   0.0000,   0.0000]],

            [[  0.2200,  -1.2136,   3.9538]],

            [[  0.8433,  -0.8104,   7.7061]],

            ...,

            [[178.8831, -73.6327, -75.3119]],

            [[178.9806, -69.9422, -74.9723]],

            [[178.5770, -66.9455, -75.4157]]], dtype=torch.float64)

In [11]:
nucl.dna.ref_frames

mod_Tensor([[[ 1.0000,  0.0000,  0.0000],
             [ 0.0000,  1.0000,  0.0000],
             [ 0.0000,  0.0000,  1.0000]],

            [[ 0.7965, -0.6046, -0.0055],
             [ 0.6010,  0.7927, -0.1017],
             [ 0.0658,  0.0777,  0.9948]],

            [[ 0.1983, -0.9801, -0.0099],
             [ 0.9750,  0.1983, -0.1002],
             [ 0.1002,  0.0102,  0.9949]],

            ...,

            [[ 0.0148, -0.9872, -0.1591],
             [ 0.0399, -0.1584,  0.9866],
             [-0.9991, -0.0209,  0.0371]],

            [[-0.6295, -0.7678, -0.1189],
             [-0.1391, -0.0391,  0.9895],
             [-0.7644,  0.6395, -0.0822]],

            [[-0.9894, -0.1390, -0.0409],
             [-0.0342, -0.0502,  0.9982],
             [-0.1408,  0.9890,  0.0449]]], dtype=torch.float64)

### Monte Carlo Simulations


System preparation


In [12]:
en = pynamod.Energy(K_bend=1)
en.set_energy_matrices(nucl, ignore_neighbors=20)

In [13]:
nucl.dna.transfer_trajectory_to_h5('test_traj.h5', 'w')

Modelling launch


In [14]:
intg = pynamod.Iterator(nucl, en, sigma_rot=0.1, sigma_transl=0.1)

In [15]:
intg.run(target_accepted_steps=1000, max_steps=10000, device='cuda', KT_factor=0.5, save_every=1, transfer_to_memory_every=100)

Steps:   0%|          | 0/10000 [00:00<?, ?it/s]

Acceptance rate:   0%|          

Start time: 03/26/26 04:37:56
target accepted steps reached
Finish time: 03/26/26 04:38:17
it/s: 28.90
accepted steps: 1000
total steps: 1446
acceptance rate: 69.16%


Visualize trajectory


In [19]:
%%time

u = nucl.get_cg_mda_traj()

CPU times: user 2min 18s, sys: 1.7 s, total: 2min 20s
Wall time: 2min 23s


/trinity/home/v_sidorov/.conda/envs/pynamod/lib/python3.13/site-packages/MDAnalysis/analysis/align.py:1773: SelectionWarning: Atoms could not be matched since they don't contain masses.
  warnings.warn(msg, category=SelectionWarning)


In [17]:
view = nv.show_mdanalysis(u)
view.clear()
view.add_representation('spacefill', radius=10)
view

NGLWidget(max_frame=1000)